# 07 — Custom Model: Train on Proprietary Claim Forms

**Time**: ~15 minutes · **Custom neural model** · **Requires labeled training data**

---

## Insurance Scenario

Your company has **proprietary claim forms** that don't match any prebuilt model. For example:

- Auto accident claim forms with company-specific fields
- Property damage assessment forms
- Internal underwriting checklists

Custom models let you train on your own labeled data to extract exactly the fields you need.

### Two Custom Model Types

| Type | Best For | Training Data |
|---|---|---|
| **Custom Template** | Fixed-layout forms (same format every time) | 5+ labeled samples |
| **Custom Neural** | Variable-layout documents (same info, different formats) | 5+ labeled samples |

> **Recommendation**: Start with **neural** — it generalizes better across layout variations.

---

## Prerequisites

1. Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb)
2. **Training data** in Azure Blob Storage with a SAS URL
3. **Labels** created using [Document Intelligence Studio](https://documentintelligence.ai.azure.com/studio)

### How to Prepare Training Data

1. Collect 5–50 samples of your claim form (more = better accuracy)
2. Upload to Azure Blob Storage
3. Open [Document Intelligence Studio](https://documentintelligence.ai.azure.com/studio) → Custom models → Create project
4. Label fields on each document (e.g., `claim_number`, `incident_date`, `claimant_name`, `damage_amount`)
5. Copy the container SAS URL for use below

In [ ]:
import os
import uuid
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import (
    DocumentIntelligenceClient,
    DocumentIntelligenceAdministrationClient,
)
from azure.ai.documentintelligence.models import (
    BuildDocumentModelRequest,
    DocumentBuildMode,
    AzureBlobContentSource,
    DocumentModelDetails,
    AnalyzeDocumentRequest,
    AnalyzeResult,
)

from azure.identity import DefaultAzureCredential

load_dotenv(dotenv_path=os.path.join("..", ".env"))

endpoint = os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"]
_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

# Administration client for model management
admin_client = DocumentIntelligenceAdministrationClient(
    endpoint=endpoint, credential=credential
)

# Analysis client for document analysis
client = DocumentIntelligenceClient(
    endpoint=endpoint, credential=credential
)

print("✅ Clients ready")

## Step 1: Check Your Resource Capacity

In [ ]:
resource_info = admin_client.get_resource_details()

print("Resource Capacity:")
print(f"  Custom models: {resource_info.custom_document_models.count} / {resource_info.custom_document_models.limit}")

if resource_info.custom_document_models.count >= resource_info.custom_document_models.limit:
    print("  ⚠️ You've reached the model limit. Delete unused models before training.")
else:
    print("  ✅ Capacity available for new models.")

## Step 2: Train a Custom Neural Model

This requires labeled training data in Azure Blob Storage. If you haven't prepared training data yet, see the prerequisites above.

> **Training time**: Neural models typically take 10–30 minutes to train. The v4.0 API provides 10 free hours of training per month.

In [ ]:
# ⚠️ Uncomment and run this cell only when you have training data ready
#
# container_sas_url = os.environ["DOCUMENTINTELLIGENCE_STORAGE_CONTAINER_SAS_URL"]
# model_id = f"insurance-claim-form-{uuid.uuid4().hex[:8]}"
#
# print(f"Training model '{model_id}'...")
# print("This may take 10-30 minutes for neural models.")
#
# poller = admin_client.begin_build_document_model(
#     BuildDocumentModelRequest(
#         model_id=model_id,
#         build_mode=DocumentBuildMode.NEURAL,  # or TEMPLATE for fixed layouts
#         azure_blob_source=AzureBlobContentSource(
#             container_url=container_sas_url,
#         ),
#         description="Insurance claim form extraction model",
#     )
# )
# model: DocumentModelDetails = poller.result()
#
# print(f"\n✅ Model trained successfully!")
# print(f"   Model ID: {model.model_id}")
# print(f"   Created: {model.created_date_time}")
# print(f"   Expires: {model.expiration_date_time}")
#
# if model.doc_types:
#     for name, doc_type in model.doc_types.items():
#         print(f"\n   Document type: {name}")
#         if doc_type.field_schema:
#             print("   Fields:")
#             for field_name, field_info in doc_type.field_schema.items():
#                 confidence = doc_type.field_confidence.get(field_name, 0) if doc_type.field_confidence else 0
#                 print(f"     - {field_name}: {field_info['type']} (accuracy: {confidence:.2%})")

print("Uncomment the code above when you have training data in Azure Blob Storage.")
print("\nTo prepare training data:")
print("1. Open https://documentintelligence.ai.azure.com/studio")
print("2. Create a Custom model project")
print("3. Upload and label your claim forms")
print("4. Copy the container SAS URL to your .env file")

## Step 3: Analyze Documents with Your Custom Model

In [ ]:
# ⚠️ Replace with your custom model ID after training
#
# custom_model_id = "insurance-claim-form-xxxxxxxx"  # Your model ID
# document_path = "../sample-documents/your-claim-form.pdf"
#
# with open(document_path, "rb") as f:
#     poller = client.begin_analyze_document(model_id=custom_model_id, body=f)
# result: AnalyzeResult = poller.result()
#
# if result.documents:
#     for doc in result.documents:
#         print(f"Document type: {doc.doc_type}")
#         print(f"Confidence: {doc.confidence:.2%}")
#         print("\nExtracted fields:")
#         if doc.fields:
#             for name, field in doc.fields.items():
#                 value = field.get("valueString") or field.get("content", "N/A")
#                 conf = field.get("confidence", 0)
#                 flag = "⚠️" if conf < 0.80 else "✅"
#                 print(f"  {flag} {name:30s} {str(value):30s} ({conf:.0%})")

print("Uncomment the code above after training a custom model (Step 2).")

## Step 4: List and Manage Models

In [ ]:
print("Your custom models:")
print("=" * 70)

models = admin_client.list_models()
count = 0
for model in models:
    # Only show non-prebuilt models
    if not model.model_id.startswith("prebuilt-"):
        count += 1
        print(f"  {model.model_id:40s} | {model.description or 'No description'}")

if count == 0:
    print("  No custom models found. Train one using Step 2 above.")
else:
    print(f"\nTotal custom models: {count}")

---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Custom neural model** | Extract fields from ANY proprietary form, even with layout variations |
| **Custom template model** | High accuracy on fixed-layout forms (same format every time) |
| **Field-level accuracy** | Know exactly how well each field is being extracted |
| **Document Intelligence Studio** | Visual labeling tool — no code required for data prep |

## Best Practices for Insurance Form Training

1. **Label 5+ samples per form variation** — more samples = better accuracy
2. **Include variations** — different handwriting, different scanners, rotated pages
3. **Use descriptive field names** — `claim_number`, `incident_date`, not `field1`, `field2`
4. **Start with neural mode** — it handles layout variations better
5. **Test with held-out samples** — don't just test on training documents

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [08-document-classifier.ipynb](08-document-classifier.ipynb) | Auto-classify incoming documents before extraction |
| [10-human-in-the-loop.ipynb](10-human-in-the-loop.ipynb) | Route low-confidence extractions to human reviewers |